In [1]:
import pandas as pd
import numpy as np
import os
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.base import clone

# 7 種指定模型
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

SIZE_GRID = {
    'mimic3c': [100, 500, 6000],
}

In [ ]:
def make_imbalanced_subset(X, y, n_total, imbalance_ratio=0.1, random_state=42):
    """
    從完整的二元資料中抽出大小為 n_total 的子集，強制使少數類佔 imbalance_ratio 比例。
    """
    n_minority = int(n_total * imbalance_ratio)
    n_majority = n_total - n_minority
    
    minority_idx = np.where(y == 1)[0]
    majority_idx = np.where(y == 0)[0]
    
    if len(minority_idx) < n_minority or len(majority_idx) < n_majority:
        raise ValueError("資料量不足以進行指定大小與比例的抽樣")
        
    np.random.seed(random_state)
    sampled_minority = np.random.choice(minority_idx, n_minority, replace=False)
    sampled_majority = np.random.choice(majority_idx, n_majority, replace=False)
    
    combined_idx = np.concatenate([sampled_minority, sampled_majority])
    np.random.shuffle(combined_idx) # 抽樣後打散順序
    
    if isinstance(X, pd.DataFrame):
        X_sub = X.iloc[combined_idx].copy()
        y_sub = y.iloc[combined_idx].copy()
    else:
        X_sub = X[combined_idx]
        y_sub = y[combined_idx]
        
    return X_sub, y_sub


In [ ]:
def load_mimic3c(use_solution_a=True):
    """
    載入 MIMIC3C 資料集。
    若 use_solution_a=True，讀取 HW1 的 df_model_ready.csv（方案 A）。
    否則讀取 mimic3c.csv 自行前處理（方案 B）。
    """
    leakage_cols = ['LOSdays', 'LOSgroupNum', 'AdmitDiagnosis']
    
    if use_solution_a:
        df = pd.read_csv('data/df_model_ready.csv')
        # 防禦性移除標籤洩漏欄位
        df = df.drop(columns=[col for col in leakage_cols if col in df.columns])
        y = df['ExpiredHospital']
        X = df.drop(columns=['ExpiredHospital'])
    else:
        df = pd.read_csv('data/mimic3c.csv')
        id_cols = ['hadm_id']
        df = df.drop(columns=leakage_cols + id_cols, errors='ignore')
        
        # 簡易類別轉換與缺失值處理
        cat_cols = df.select_dtypes(include=['object']).columns
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
        df = df.fillna(df.median())
        
        y = df['ExpiredHospital']
        X = df.drop(columns=['ExpiredHospital'])
        
    print(f"X shape: {X.shape}, y=1 比例: {y.mean():.4f}, 最終納入特徵數量: {X.shape[1]}")
    return X, y